# OneDrive → Google Drive migration

Run the cells in order. When prompted, upload the `rclone.conf` created by `setup-auth.ps1`.

The destination is `My Drive/OneDrive Migration`. Rerunning is safe: this uses `rclone copy`, not `sync`.


In [ ]:
import shutil
import subprocess

if shutil.which("rclone") is None:
    subprocess.run(["bash", "-lc", "curl -fsSL https://rclone.org/install.sh | sudo bash"], check=True)

subprocess.run(["rclone", "version"], check=True)


In [ ]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Upload exactly one rclone.conf file.")

name, data = next(iter(uploaded.items()))
if Path(name).name.lower() != "rclone.conf":
    raise RuntimeError("The uploaded file must be named rclone.conf.")

CONFIG = "/content/rclone.conf"
Path(CONFIG).write_bytes(data)
Path(CONFIG).chmod(0o600)
print("rclone.conf loaded into this Colab runtime only.")


In [ ]:
SOURCE = "onedrive-src:"
DESTINATION = "gdrive-dst:OneDrive Migration"

def run_rclone(*args, check=True):
    command = ["rclone", *args, "--config", CONFIG]
    return subprocess.run(command, check=check)

configured = subprocess.run(
    ["rclone", "listremotes", "--config", CONFIG],
    check=True, capture_output=True, text=True
).stdout.splitlines()

for required in ("onedrive-src:", "gdrive-dst:"):
    if required not in configured:
        raise RuntimeError(f"Missing required remote: {required}")

print("Validating OneDrive access...")
run_rclone("lsf", SOURCE, "--max-depth", "1")
print("Validating Google Drive access...")
run_rclone("lsf", "gdrive-dst:", "--max-depth", "1")
print("Both remotes are accessible.")


In [ ]:
print(f"Copying {SOURCE} -> {DESTINATION}")
run_rclone(
    "copy", SOURCE, DESTINATION,
    "--progress",
    "--stats", "10s",
    "--transfers", "4",
    "--checkers", "8",
    "--create-empty-src-dirs",
)
print("Copy finished. Starting verification...")


In [ ]:
REPORT = "/content/rclone-check.txt"
result = run_rclone(
    "check", SOURCE, DESTINATION,
    "--one-way",
    "--size-only",
    "--combined", REPORT,
    "--checkers", "8",
    check=False,
)

if result.returncode != 0:
    print(Path(REPORT).read_text(errors="replace") if Path(REPORT).exists() else "No report produced.")
    raise RuntimeError(f"Verification FAILED (rclone exit code {result.returncode}).")

print("Verification SUCCESS: every OneDrive file is present at the destination with the same size.")
